# 06 · Findings — H0–H5 verdicts (feeds §8 / FINDINGS)

Consolidates each hypothesis to an explicit verdict using the §8 rules (noise floor first; per-cell, no sign-flip averaging). Prints text ready to paste into the paper's Discussion.

In [1]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))  # RKD/ on path
sys.path.insert(0, os.getcwd())
import analysis_utils as au
import pandas as pd
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

In [2]:
df = pd.read_csv('results.csv')
lines = []
# H0
g1 = df[(df.phase=='phase1')&(df.student=='graph-rkd')]
floor = df[(df.phase=='phase1')&(df.student=='triplet_only')]['val_mAP@R'].mean()
if len(g1):
    band = au.agg(g1,['lambda_g'],'val_mAP@R'); viable = band[band['mean']>=floor]
    lines.append(f"H0: {'ACCEPT' if len(viable) else 'REJECT'} — viable λg: {list(viable.lambda_g)} (floor={floor:.4f})")
else: lines.append('H0: sem dados (phase1)')
# H1 por célula
for d,t in df[df.phase=='phase5'][['dataset','teacher']].dropna().drop_duplicates().itertuples(index=False):
    lines.append(f'H1 [{d}·{t}]: '+au.h1_verdict(au.headline_table(df,d,t)))
print('\n'.join(lines) if lines else 'sem runs ainda')

H0: ACCEPT — viable λg: [0.01] (floor=0.0232)
H1 [cars196·resnet18]: Graph-RKD VENCE triplet-only (mediana 0.0346 > 0.0307)
H1 [cars196·convnext_tiny]: Graph-RKD VENCE triplet-only (mediana 0.0351 > 0.0307)
H1 [cub200·resnet18]: Graph-RKD PERDE p/ +RKD-A (mediana 0.0294 < 0.0361)
H1 [cub200·convnext_tiny]: Graph-RKD PERDE p/ +RKD-A (mediana 0.0300 < 0.0312)


### H3 mechanism (available now from the probe)

In [3]:
p = au.load_probe()
print('MDS near-degenerate rate by N:')
print(p[p.method=='mds'].groupby('N')['mds_degenerate_rate'].max())
print('\nprofile tie rate by N:')
print(p[p.method=='profile'].groupby('N')['profile_tie_rate'].max())

MDS near-degenerate rate by N:
N
3    0.0013
4    0.0067
8    0.1267
16   0.9040
17   0.9460
Name: mds_degenerate_rate, dtype: float64

profile tie rate by N:
N
3    0.0342
4    0.0553
8    0.1479
16   0.3064
17   0.3209
Name: profile_tie_rate, dtype: float64
